# Task 4 — Search Test Platform

This notebook shows one safe visual-search run:

`image -> crop -> letterbox -> R5 embedding -> Top-K`

This demo uses only development images or a new outside image. It does not access holdout images, labels, or gallery. The project holdout was opened once for final evaluation and is now permanently closed. Quarantine and official teacher-test data also remain closed to this demo.

In [ ]:
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mla2-matplotlib")

from pathlib import Path

import pandas as pd
from IPython.display import Image as DisplayImage
from IPython.display import display

from fashion.config import ROOT
from fashion.task4 import CropBox, load_search_bundle, run_search, write_search_outputs

## Choose one query mode

Keep `OUTSIDE_IMAGE` as `None` to search with the safe fold-1 development image below. Or set it to a new outside image and set `KNOWN_QUERY_ID` to `None`.

The geometry warnings report unusual shape or heavy padding. They do not prove that an image was stretched. Use `CROP` only when you want to remove a border or background before resizing.

In [ ]:
KNOWN_QUERY_ID = 1529
OUTSIDE_IMAGE: Path | None = None
CROP: CropBox | None = None
TOP_K = 5
RATING: str | None = None
NOTE: str | None = None

In [ ]:
bundle = load_search_bundle(
    model_package=ROOT / "models/task4_r5",
    gallery_directory=ROOT / "models/task4_teacher_gallery",
    splits_path=ROOT / "data/processed/splits.csv",
    device="cpu",
)

if OUTSIDE_IMAGE is None:
    response = run_search(
        bundle,
        query_id=KNOWN_QUERY_ID,
        crop=CROP,
        top_k=TOP_K,
    )
else:
    response = run_search(
        bundle,
        image_path=OUTSIDE_IMAGE,
        crop=CROP,
        top_k=TOP_K,
        rating=RATING,
        note=NOTE,
    )

In [ ]:
query = response.record.query
query_facts = pd.DataFrame(
    [
        {
            "kind": query.kind,
            "known_id": query.known_id,
            "source_dimensions": query.source_dimensions,
            "effective_dimensions": query.effective_dimensions,
            "aspect_ratio": query.aspect_ratio,
            "content_fraction": query.content_fraction,
            "crop": None if query.crop is None else query.crop.to_dict(),
            "warnings": list(query.warnings),
        }
    ]
)
display(query_facts)

In [ ]:
top_k_results = pd.DataFrame(
    [hit.to_dict() for hit in response.record.results]
)
display(top_k_results)

In [ ]:
figure_path, evidence_path = write_search_outputs(
    response,
    figure_directory=ROOT / "results/figures/task4/test-platform",
    evidence_directory=ROOT / "results/evidence/task4/test-platform",
)
print(f"Figure: {figure_path.relative_to(ROOT)}")
print(f"Evidence: {evidence_path.relative_to(ROOT)}")
display(DisplayImage(filename=str(figure_path)))